# Recursive fold + a model that predicts the shape's amplitude (and position)

Prototype of the proposed design, **no token changes** -- pure numpy so you can
validate the idea first.

Plan:
1. **One adaptive fold** `fold(series, period)` -> a frozen `shape` + an
   `amplitude` (one number per cycle). Apply it **recursively**, as many times
   as there are periods (week -> month -> ...).
2. **The model only forecasts the amplitude trajectory** (a smooth, low-D
   series), then we rebuild the full forecast by multiplying the shapes back.
3. **Position / phase**: if the shape drifts in time, amplitude alone fails;
   predicting a per-cycle *phase* fixes it. This is the 'amplitude OR position'
   idea, shown side by side.


## 0. Helpers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=3, suppress=True)

def fold(series, period):
    """Fold by `period` -> mean-1 frozen shape + one amplitude per cycle."""
    n = len(series) // period
    M = series[-n*period:].reshape(n, period)
    mean = M.mean(1, keepdims=True); mean = np.where(mean == 0, 1.0, mean)
    shape = (M / mean).mean(0); shape = shape / shape.mean()   # mean-1 profile
    amp   = (M / shape[None, :]).mean(1)                        # 1 amplitude / cycle
    return shape, amp

def decompose(series, periods):
    """Apply fold recursively. Returns the shape at each level (apply order) and
    the innermost amplitude (the smooth 'trend')."""
    shapes = []; cur = series
    for p in periods:
        s, a = fold(cur, p); shapes.append(s); cur = a
    return shapes, cur

def rebuild(top_amps, shapes, periods):
    """Expand a top-level amplitude series back to full resolution by
    multiplying the shapes (outer -> inner)."""
    cur = np.asarray(top_amps, float)
    for p, s in zip(reversed(periods), reversed(shapes)):
        cur = (cur[:, None] * s[None, :]).reshape(-1)
    return cur

def mape(p, t):
    return float(np.mean(np.abs(p - t) / np.abs(t)) * 100)


## 1. Data: daily series = trend(month) x week-of-month x day-of-week

In [ ]:
DOW   = np.array([1.0,1.1,1.2,1.0,1.3,0.6,0.5]); DOW = DOW/DOW.mean()
WOM   = np.array([1.2,1.0,0.9,0.9])
TRAIN_MONTHS = 6
def month_level(m): return 1000*(1+0.05*m)

full = np.concatenate([month_level(m)*WOM[w]*DOW
                       for m in range(TRAIN_MONTHS+1) for w in range(4)])
train, future = full[:TRAIN_MONTHS*28], full[TRAIN_MONTHS*28:]    # predict next 28 days

plt.figure(figsize=(13,3.5))
plt.plot(np.arange(len(train)), train, label="history")
plt.plot(np.arange(len(train), len(full)), future, color="black", lw=2, label="true future")
plt.axvline(len(train), color="gray", ls="--"); plt.title("daily series"); plt.legend(); plt.show()


## 2. Recursive decomposition (week -> month)

`periods=[7,4]`: fold the daily series by 7, then fold the resulting weekly
amplitude by 4. We get two frozen shapes and a smooth monthly trend.

In [ ]:
PERIODS = [7, 4]
shapes, trend = decompose(train, PERIODS)
print("shape level 1 (day-of-week):", shapes[0])
print("shape level 2 (week-of-month):", shapes[1])
print("trend (innermost amplitude):", trend.round(1))

fig, ax = plt.subplots(1,3, figsize=(14,3.2))
ax[0].bar(range(7), shapes[0]); ax[0].set_title("shape: day-of-week"); ax[0].axhline(1,color="k",lw=.6)
ax[1].bar(range(4), shapes[1]); ax[1].set_title("shape: week-of-month"); ax[1].axhline(1,color="k",lw=.6)
ax[2].plot(trend, "-o"); ax[2].set_title("trend = what the MODEL forecasts"); ax[2].set_xlabel("month")
plt.tight_layout(); plt.show()


## 3. The model forecasts ONLY the amplitude (trend); we rebuild the rest

Compare against doing just **one** fold (weekly) -- which leaves the monthly
swing unmodeled.

In [ ]:
H_TOP = 1   # forecast 1 top-level cycle (1 month = 28 days)

# two folds: forecast the smooth monthly trend, expand through both shapes
sl, ic = np.polyfit(np.arange(len(trend)), trend, 1)
trend_fc = np.array([sl*(len(trend)+k)+ic for k in range(H_TOP)])
fc_two = rebuild(trend_fc, shapes, PERIODS)

# one fold only: forecast the weekly amplitude directly (still wiggly), expand 1 shape
sh1, amp1 = fold(train, 7)
s1, i1 = np.polyfit(np.arange(len(amp1)), amp1, 1)
amp1_fc = np.array([s1*(len(amp1)+w)+i1 for w in range(4)])
fc_one = rebuild(amp1_fc, [sh1], [7])

print(f"MAPE one fold (weekly only)      : {mape(fc_one, future):.2f}%")
print(f"MAPE two folds (weekly + monthly): {mape(fc_two, future):.2f}%")

t = np.arange(28)
plt.figure(figsize=(13,4))
plt.plot(t, future, color="black", lw=2, label="true future")
plt.plot(t, fc_one, "--", label=f"1 fold  ({mape(fc_one,future):.1f}%)")
plt.plot(t, fc_two, "--", label=f"2 folds ({mape(fc_two,future):.1f}%)")
plt.title("model only forecasts the amplitude; shapes rebuild the rest"); plt.legend(); plt.show()


## 4. Why it's simpler: what the model actually sees

Left: the raw daily series (what a naive model must learn). Right: the trend
the model in this scheme has to forecast -- a near-line.

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(13,3.2))
ax[0].plot(train); ax[0].set_title("raw daily series (hard: 2 seasonalities + trend)")
ax[1].plot(trend, "-o"); ax[1].set_title("amplitude the model forecasts (easy: ~a line)")
plt.tight_layout(); plt.show()


## 5. Position / phase: when the shape DRIFTS in time

Now make the weekly peak slowly shift later each week. Amplitude-only freezes
the shape and fails. Estimating a per-cycle **phase** (and forecasting it) lets
the shape rotate -- the 'predict the position of the shape' idea.

In [ ]:
def roll_frac(s, shift):                       # circular shift by fractional samples
    n=len(s); k=np.fft.rfftfreq(n,1/n); F=np.fft.rfft(s)
    return np.fft.irfft(F*np.exp(-1j*2*np.pi*k*shift/n), n)
def fft_phase(c): return np.angle(np.fft.rfft(c-c.mean())[1])   # phase of fundamental
def ph_to_shift(phase, period): return -phase*period/(2*np.pi)  # phase(rad)->samples

period=7; nweeks=24; drift=0.15        # radians of weekly peak-shift per week
base=np.array([1.0,1.1,1.2,1.0,1.3,0.6,0.5]); base/=base.mean()
amp =np.array([1000*(1+0.01*w) for w in range(nweeks)])
sh  =lambda w: drift*w*period/(2*np.pi)
train2=np.concatenate([amp[w]*roll_frac(base, sh(w)) for w in range(nweeks)])
H=4
future2=np.concatenate([(1000*(1+0.01*(nweeks+h)))*roll_frac(base, sh(nweeks+h)) for h in range(H)])

M=train2.reshape(nweeks,7); amp_hat=M.mean(1)
ph=np.unwrap([fft_phase(M[w]) for w in range(nweeks)])          # per-week phase
sa,sb=np.polyfit(np.arange(nweeks),amp_hat,1)                   # amplitude trend
pa,pb=np.polyfit(np.arange(nweeks),ph,1)                        # PHASE trend

# amplitude-only: average the raw (now SMEARED) shapes, freeze it
smear=(M/amp_hat[:,None]).mean(0); smear/=smear.mean()
fc_amp=np.concatenate([(sa*(nweeks+h)+sb)*smear for h in range(H)])

# amplitude + phase: de-rotate each cycle to phase 0 -> sharp shape, then re-rotate by forecast phase
aligned=np.stack([roll_frac(M[w]/amp_hat[w], -ph_to_shift(ph[w],period)) for w in range(nweeks)])
S0=aligned.mean(0); S0/=S0.mean()
fc_ph=np.concatenate([(sa*(nweeks+h)+sb)*roll_frac(S0, ph_to_shift(pa*(nweeks+h)+pb, period)) for h in range(H)])

print(f"amplitude-only  (frozen shape) MAPE: {mape(fc_amp, future2):.2f}%")
print(f"amplitude+phase (rotating shape) MAPE: {mape(fc_ph, future2):.2f}%")

fig, ax = plt.subplots(1,2, figsize=(14,3.6))
ax[0].plot(np.arange(nweeks), ph, "-o"); ax[0].set_title("estimated weekly PHASE (drifts -> forecastable)")
ax[0].set_xlabel("week")
t=np.arange(H*7)
ax[1].plot(t, future2, color="black", lw=2, label="true")
ax[1].plot(t, fc_amp, "--", label=f"amplitude only ({mape(fc_amp,future2):.0f}%)")
ax[1].plot(t, fc_ph,  "--", label=f"amplitude+phase ({mape(fc_ph,future2):.0f}%)")
ax[1].set_title("drifting shape: phase prediction fixes it"); ax[1].legend()
plt.tight_layout(); plt.show()


## 6. Takeaway for the token design

- `fold(series, period)` is the single adaptive primitive; apply it recursively
  (`SeasonalFold` token, once per detected period).
- The model's target collapses to the **amplitude trajectory** (a near-line) and
  optionally a **phase trajectory** -- both tiny and smooth, the 'much simpler
  form'. Reconstruction is just multiplying the shapes back (a decode step).
- Amplitude handles seasonality that scales; **phase** handles seasonality whose
  pattern shifts in time -- which frozen-shape FLAIR cannot. The two are separate
  things the model can predict.
